# RoadAlert AI Training Suite (DL, NLP, ML)
### Intelligent Traffic Incident Verification & Road Condition Platform

This notebook provides separate, decoupled AI pipelines for:
1. **Deep Learning (DL):** Road Damage Detection (RDD 8-class defect detection from `data/RoadDamageDetector`).
2. **Natural Language Processing (NLP):** Incident report synthesis directly loaded from Hugging Face (`DSTI/traffic-accidents-reports-800`).
3. **Machine Learning (ML):** Tabular incident verification and explainable confidence scoring aligned with RoadAlert's Section 10 verification engine.

> **Important Note:** Each pipeline addresses a distinct capability. Deep learning identifies physical surface distress (potholes, cracks), NLP formats incident narratives, and ML assesses multi-source evidence and corroboration. Physical surface damage does not prove an incident is genuine or recent.

## 1. Environment & Project Root Resolution
We resolve the project root dynamically so this notebook runs seamlessly from VS Code, JupyterLab, or command line.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve project root reliably
cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "artifacts").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "artifacts").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "roadalert_ai"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"[RoadAlert AI] Project Root: {PROJECT_ROOT}")
print(f"[RoadAlert AI] Artifacts Path: {ARTIFACTS_DIR}")
print(f"[RoadAlert AI] Python: {sys.version.split()[0]}")

## 2. Deep Learning (DL) — Road Damage Detection
- **Dataset Path:** `data/RoadDamageDetector` (relative to project root)
- **Annotation Format:** Pascal VOC XML annotations (`<annotation><object><name>D40</name>...`)
- We inspect the dataset structure, parse classes from `crackLabelMap.txt`, and configure the object detection pipeline.

In [ ]:
import re
import json

dl_dir = PROJECT_ROOT / "data" / "RoadDamageDetector"
label_map_file = dl_dir / "crackLabelMap.txt"
images_dir = dl_dir / "images"

print(f"Checking DL Directory: {dl_dir}")
assert dl_dir.exists(), f"Directory not found: {dl_dir}"

dl_classes = {}
if label_map_file.exists():
    content = label_map_file.read_text(encoding="utf-8")
    items = re.findall(r"item\s*\{[^}]*id:\s*(\d+)[^}]*name:\s*['\"]([^'\"]+)['\"][^}]*\}", content)
    for cid, cname in items:
        dl_classes[int(cid)] = cname

class_descriptions = {
    "D00": "Wheel Mark Longitudinal Crack",
    "D01": "Construction Joint Longitudinal Crack",
    "D10": "Equal Interval Lateral Crack",
    "D11": "Construction Joint Lateral Crack",
    "D20": "Alligator (Fatigue) Mesh Crack",
    "D40": "Rutting, Bump, Pothole, Separation",
    "D43": "Crosswalk / White Line Blur",
    "D44": "Lane Mark / Center Line Blur"
}

print(f"Detected {len(dl_classes)} Road Damage Classes from crackLabelMap.txt:")
for cid, code in sorted(dl_classes.items()):
    print(f"  [{cid}] {code}: {class_descriptions.get(code, 'Defect')}")

images = list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) if images_dir.exists() else []
print(f"\nDetected {len(images)} images in {images_dir}")
for img in images:
    print(f"  - {img.name} ({img.stat().st_size / 1024:.1f} KB)")

### 2.1 DL Dataset Splitting & Data Leakage Prevention
When working with road damage datasets collected from vehicle-mounted cameras:
- **Random splitting causes severe data leakage** because successive video frames have near-identical backgrounds.
- **Correct approach:** Group frames by road segment / recording session (80% train, 10% validation, 10% test).

In [ ]:
# Object Detection Architecture Configuration
dl_config = {
    "architecture": "YOLOv8-Small / Faster R-CNN (MobileNetV3 Backbone)",
    "input_size": [640, 640],
    "num_classes": len(dl_classes),
    "classes": [{"id": k, "code": v, "description": class_descriptions.get(v, "Defect")} for k, v in dl_classes.items()],
    "split": {"train": 0.80, "val": 0.10, "test": 0.10, "grouping": "by_road_sequence"},
    "target_metrics": {"mAP@0.5": 0.65, "mAP@0.5:0.95": 0.40, "inference_time_ms": 20}
}

with open(ARTIFACTS_DIR / "dl_model_config.json", "w", encoding="utf-8") as f:
    json.dump(dl_config, f, indent=2)

print("DL model configuration saved to artifacts/roadalert_ai/dl_model_config.json")

## 3. Natural Language Processing (NLP) — Accident Reports 800
We load `DSTI/traffic-accidents-reports-800` directly from the Hugging Face Hub using the `datasets` library.

In [ ]:
from datasets import load_dataset

print("Loading 'DSTI/traffic-accidents-reports-800' from Hugging Face...")
dataset = load_dataset("DSTI/traffic-accidents-reports-800")

print("\nDataset splits:", list(dataset.keys()))
for split_name, split_data in dataset.items():
    print(f"  - {split_name}: {len(split_data)} rows | Columns: {split_data.column_names}")

sample = dataset["train"][0]
print("\n--- Sample Input (5W1H) ---")
print(sample["input"])
print("\n--- Sample Target (Report Narrative) ---")
print(sample["target"])

### 3.1 NLP Baseline: 5W1H Extraction & Report Synthesis
We implement an explainable, structured synthesis baseline and evaluate it against the official evaluation split using ROUGE-1, ROUGE-2, and ROUGE-L metrics.

In [ ]:
from collections import Counter

def parse_5w1h(text: str) -> dict:
    fields = {}
    for line in text.strip().split("\n"):
        if ":" in line:
            k, v = line.split(":", 1)
            fields[k.strip()] = v.strip()
    return fields

def generate_report(fields: dict) -> str:
    what = fields.get("What", "A traffic collision")
    when = re.sub(r"Occurrence:\s*", "", fields.get("When", "recently")).split(";")[0].strip()
    where = fields.get("Where", "the roadway")
    who = fields.get("Who", "parties involved")
    how = fields.get("How", "in slow traffic")
    why = fields.get("Why", "traffic circumstances")
    action = fields.get("ContingencyActions", "Emergency responders responded")
    
    return f"On {when}, {what} was reported at {where}. The incident involved {who}. According to preliminary observations, {how}, influenced by {why}. Response teams took action: {action.lower()}."

def tokenize(text: str) -> list:
    return re.findall(r"\b\w+\b", text.lower())

def compute_rouge_1(cand: list, ref: list) -> float:
    c_counts = Counter(cand)
    r_counts = Counter(ref)
    overlap = sum((c_counts & r_counts).values())
    p = overlap / max(1, len(cand))
    r = overlap / max(1, len(ref))
    return 2 * p * r / max(1e-8, (p + r))

eval_data = dataset["eval"]
r1_scores = []
for row in eval_data:
    fields = parse_5w1h(row["input"])
    pred = generate_report(fields)
    score = compute_rouge_1(tokenize(pred), tokenize(row["target"]))
    r1_scores.append(score)

avg_r1 = sum(r1_scores) / len(r1_scores)
print(f"Baseline Evaluation on {len(eval_data)} Eval Split Examples:")
print(f"  ROUGE-1 F1 Score: {avg_r1:.4f}")

## 4. Machine Learning (ML) — Tabular Incident Verification
- We inspect RoadAlert's database schema (`database/schema.prisma`).
- Available records: Exactly 18 seeded demo incidents in coastal Karnataka.
- **Critical Rule:** 18 records are not enough for supervised classification. We implement Section 10's explainable heuristic verification baseline and create an audited template for collecting training data.

In [ ]:
import pandas as pd

template_file = ARTIFACTS_DIR / "incident_data_template.csv"
if template_file.exists():
    df_template = pd.read_csv(template_file)
    print("Loaded Incident Data Collection Template:")
    print(f"Shape: {df_template.shape} | Columns: {len(df_template.columns)}")
    print(df_template[["incident_id", "category", "severity", "confidence_score", "verified_label"]].head())
else:
    print("Template not yet generated.")

### 4.1 Section 10 Explainable Verification Scoring Baseline
Calculates a normalized 0-100 confidence score using multi-source corroboration:

In [ ]:
def calculate_incident_confidence(reporters: int, latency_min: float, gps_drop_kmh: float, evidences: int, authority: bool) -> dict:
    pts_reporters = min(30, reporters * 10)
    pts_latency = max(0, min(20, int(20 - (latency_min / 5))))
    pts_gps = min(20, int(gps_drop_kmh * 0.8))
    pts_evidence = min(15, int(evidences * 7.5))
    pts_authority = 15 if authority else 0
    
    total = min(100, pts_reporters + pts_latency + pts_gps + pts_evidence + pts_authority)
    band = "VERIFIED" if total >= 70 else ("CORROBORATED" if total >= 40 else "UNVERIFIED")
    
    return {
        "confidence_score": total,
        "band": band,
        "breakdown": {
            "distinct_reporters": pts_reporters,
            "temporal_proximity": pts_latency,
            "gps_telemetry": pts_gps,
            "validated_evidence": pts_evidence,
            "authority_confirmation": pts_authority
        }
    }

# Test with a multi-source verified incident
test_score = calculate_incident_confidence(
    reporters=4,
    latency_min=6.0,
    gps_drop_kmh=32.0,
    evidences=2,
    authority=True
)
print("Sample Multi-Source Confidence Breakdown:")
print(json.dumps(test_score, indent=2))

## 5. Summary of Generated Artifacts
Check that all saved models, evaluation reports, and configs exist in `artifacts/roadalert_ai/`.

In [ ]:
print("Artifacts in artifacts/roadalert_ai/:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  - {f.name} ({f.stat().st_size} bytes)")

## 6. Google Colab Full GPU Training Guide
To train the full YOLOv8 road damage model on a free Colab GPU:
```python
# In Google Colab:
!git clone https://github.com/pavandevarunaik2006-lang/nexathon26.git
%cd nexathon26
!pip install -r requirements-ai.txt
!pip install ultralytics

# Download RDD dataset
!wget https://s3-ap-northeast-1.amazonaws.com/mycityreport/RoadDamageDataset.tar.gz -O data/RoadDamageDataset.tar.gz
!tar -xzf data/RoadDamageDataset.tar.gz -C data/

# Train YOLOv8s
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
results = model.train(data='data/RoadDamageDetector/rdd_colab.yaml', epochs=50, imgsz=640, batch=16, device=0)
model.export(format='onnx')
```